# Weight Analysis Notebook

This notebook provides tools to analyze, load, and inspect pretrained model weights from ClimateSAM checkpoints. It includes utilities for examining weight statistics, layer information, and adapter components.

In [2]:
import os
import sys
import torch
import numpy as np
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    
    # Setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import model components
try:
    from model.climatesam import ClimateSAM
    print("✓ ClimateSAM imported successfully")
except ImportError as e:
    print(f"✗ Error importing ClimateSAM: {e}")

Project root: C:\Users\perrydebussy\Project\Study\masterarbeit
PyTorch version: 2.8.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5070
Using device: cuda:0
✓ ClimateSAM imported successfully


In [3]:
# Initialize model
model_type = 'vit_b'  # Change this to 'vit_l' or 'vit_h' if needed
mlp_ratio = 1.0

try:
    model = ClimateSAM(
        model_type=model_type,
        mlp_ratio=mlp_ratio,
        enable_wandb_logging=False
    ).to(device=device)
    print(f"✓ Model initialized with type: {model_type}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
except Exception as e:
    print(f"✗ Error initializing model: {e}")

✓ Model initialized with type: vit_b
Model parameters: 101,262,723


In [16]:
# Setup checkpoint paths
exp_dir = Path('exp')
# checkpoint_name = 'phase_1_weights_official_vit_b_infused_token_vit_b_mlp1.pth'
checkpoint_name = 'exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED_NOSMOOTH.pth'
image_encoder_path = exp_dir / checkpoint_name

# Verify path exists
if image_encoder_path.exists():
    print(f"✓ Checkpoint found: {image_encoder_path}")
    print(f"  File size: {image_encoder_path.stat().st_size / (1024**2):.2f} MB")
else:
    print(f"✗ Checkpoint not found at: {image_encoder_path}")
    print(f"Available files in {exp_dir}:")
    if exp_dir.exists():
        for f in sorted(exp_dir.glob('*.pth'))[:5]:
            print(f"  - {f.name}")

✓ Checkpoint found: exp\exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED_NOSMOOTH.pth
  File size: 386.44 MB


In [21]:
import os
import torch


# image_encoder_path = "exp\exp_phase_1_weights_official_vit_b_infused_token_vit_b_mlp1.pth"

# Load checkpoint
print("=" * 60)
print("LOADING CHECKPOINT")
print("=" * 60)

try:
    phase_1_checkpoint = torch.load(image_encoder_path, map_location=device)
    print(f"✓ Pretrained weights loaded from {image_encoder_path}")
except Exception as e:
    print(f"✗ Error loading checkpoint: {e}")
    raise

# Print checkpoint structure
print(f"\nCheckpoint keys: {list(phase_1_checkpoint.keys())}")
# print(f"Checkpoint size: {sum(v.numel() for v in phase_1_checkpoint.values()):,} parameters")

# Load image encoder and mask decoder
print("\n" + "=" * 60)
print("LOADING MODEL COMPONENTS")
print("=" * 60)

try:
    model.image_encoder.load_state_dict(phase_1_checkpoint['image_encoder'])
    print("✓ Image encoder weights loaded")
except Exception as e:
    print(f"✗ Error loading image encoder: {e}")

try:
    model.mask_decoder.load_state_dict(phase_1_checkpoint['mask_decoder'])
    print("✓ Mask decoder weights loaded")
except Exception as e:
    print(f"✗ Error loading mask decoder: {e}")

# Load and print input adapter weights if available
print("\n" + "=" * 60)
print("INPUT ADAPTER ANALYSIS")
print("=" * 60)

if 'input_adapter' in phase_1_checkpoint:
    print("\n✓ Input adapter found in checkpoint\n")
    input_adapter_state = phase_1_checkpoint['input_adapter']
    
    total_params = 0
    for param_name, param_value in input_adapter_state.items():
        num_params = param_value.numel()
        total_params += num_params
        print(f"{param_name}:")
        print(f"  Shape: {param_value.shape}")
        print(f"  Dtype: {param_value.dtype}")
        print(f"  Parameters: {num_params:,}")
        print(f"  Mean: {param_value.mean():.6f}")
        print(f"  Std: {param_value.std():.6f}")
        print(f"  Min: {param_value.min():.6f}")
        print(f"  Max: {param_value.max():.6f}\n")
    
    print(f"Total adapter parameters: {total_params:,}")
    
    # Load into model if it has input_adapter attribute
    if hasattr(model, 'input_adapter'):
        try:
            model.input_adapter.load_state_dict(input_adapter_state)
            print("✓ Input adapter weights loaded into model")
        except Exception as e:
            print(f"✗ Error loading input adapter into model: {e}")
    else:
        print("⚠ Model does not have 'input_adapter' attribute")
else:
    print("⚠ Input adapter not found in checkpoint")

LOADING CHECKPOINT
✓ Pretrained weights loaded from exp\exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED_NOSMOOTH.pth

Checkpoint keys: ['image_encoder', 'mask_decoder', 'input_adapter']

LOADING MODEL COMPONENTS
✓ Image encoder weights loaded
✓ Mask decoder weights loaded

INPUT ADAPTER ANALYSIS

✓ Input adapter found in checkpoint

input_adapt.0.weight:
  Shape: torch.Size([3, 16, 1, 1])
  Dtype: torch.float32
  Parameters: 48
  Mean: 0.055890
  Std: 0.253084
  Min: -0.118281
  Max: 1.018404

input_adapt.0.bias:
  Shape: torch.Size([3])
  Dtype: torch.float32
  Parameters: 3
  Mean: 0.003677
  Std: 0.013816
  Min: -0.011920
  Max: 0.014378

Total adapter parameters: 51
✓ Input adapter weights loaded into model


In [18]:
# Print only specific adapter layer weights
variables = ['TMQ', 'U850', 'V850', 'UBOT', 'VBOT', 'QREFHT', 'PS', 'PSL', 
                    'T200', 'T500', 'PRECT', 'TS', 'TREFHT', 'Z1000', 'Z200', 'ZBOT']
channel = ['R', 'G', 'B']
if 'input_adapter' in phase_1_checkpoint:
    adapter_weights = phase_1_checkpoint['input_adapter']
    for name, param in adapter_weights.items():
        for i in range (0, 3):
            if 'weight' in name:  # Only print weight parameters, not biases
                print(f"{name}: {param.shape}")
                
                ch = channel[i]
                weight = param.flatten()[i:i+16]
                info = f"  {name} - Channel: {ch}"
                for i in range(16):
                    info += f"\n    {variables[i]}: {weight[i]:.6f}"
                print(info)


            
            

input_adapt.0.weight: torch.Size([3, 16, 1, 1])
  input_adapt.0.weight - Channel: R
    TMQ: 1.018404
    U850: 0.023962
    V850: 0.013941
    UBOT: -0.011747
    VBOT: -0.035003
    QREFHT: 0.034553
    PS: -0.025955
    PSL: -0.049943
    T200: -0.042053
    T500: -0.040428
    PRECT: 0.083689
    TS: 0.033044
    TREFHT: -0.078597
    Z1000: 0.014781
    Z200: -0.061811
    ZBOT: -0.008791
input_adapt.0.weight: torch.Size([3, 16, 1, 1])
  input_adapt.0.weight - Channel: G
    TMQ: 0.023962
    U850: 0.013941
    V850: -0.011747
    UBOT: -0.035003
    VBOT: 0.034553
    QREFHT: -0.025955
    PS: -0.049943
    PSL: -0.042053
    T200: -0.040428
    T500: 0.083689
    PRECT: 0.033044
    TS: -0.078597
    TREFHT: 0.014781
    Z1000: -0.061811
    Z200: -0.008791
    ZBOT: -0.013886
input_adapt.0.weight: torch.Size([3, 16, 1, 1])
  input_adapt.0.weight - Channel: B
    TMQ: 0.013941
    U850: -0.011747
    V850: -0.035003
    UBOT: 0.034553
    VBOT: -0.025955
    QREFHT: -0.049943
  

LORA

In [ ]:
model = ClimateSAM(
    model_type=model_type,
    mlp_ratio=mlp_ratio,
    enable_wandb_logging=False
).to(device=device)
def print_adapter_weights(checkpoint_path, model):
        # Print only specific adapter layer weights
    variables = ['TMQ', 'U850', 'V850', 'UBOT', 'VBOT', 'QREFHT', 'PS', 'PSL', 
                        'T200', 'T500', 'PRECT', 'TS', 'TREFHT', 'Z1000', 'Z200', 'ZBOT']
    channels = ['R', 'G', 'B']
    for name, param in model.input_adapter.named_parameters():
            if 'weight' in name:
                print(f"\n{name}: {param.shape}") # Should be [3, 16, 1, 1]
                
                for out_idx in range(3):
                    ch_name = channels[out_idx]
                    # Access [output_channel, :, 0, 0] to get all 16 input weights
                    ch_weights = param[out_idx, :, 0, 0] 
                    
                    print(f"  Channel: {ch_name}")
                    for var_idx in range(len(variables)):
                        val = ch_weights[var_idx].item()
                        print(f"    {variables[var_idx]}: {val:.6f}")

    print(f"✓ Model initialized with type: {model_type}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    phase_1_checkpoint = torch.load(checkpoint_path, map_location=device)
    try:
        model.image_encoder.load_state_dict(phase_1_checkpoint['image_encoder'])
        print("✓ Image encoder weights loaded")
    except Exception as e:
        print(f"✗ Error loading image encoder: {e}")

    try:
        model.mask_decoder.load_state_dict(phase_1_checkpoint['mask_decoder'])
        print("✓ Mask decoder weights loaded")
    except Exception as e:
        print(f"✗ Error loading mask decoder: {e}")
    if 'input_adapter' in phase_1_checkpoint:
        print("\n✓ Input adapter found in checkpoint\n")
        input_adapter_state = phase_1_checkpoint['input_adapter']
        model.input_adapter.load_state_dict(input_adapter_state)
    
    if 'input_adapter' in phase_1_checkpoint:
        adapter_weights = phase_1_checkpoint['input_adapter']
        for name, param in adapter_weights.items():
            if 'weight' in name:
                print(f"\n{name}: {param.shape}") # Should be [3, 16, 1, 1]
                
                for out_idx in range(3):
                    ch_name = channels[out_idx]
                    # Access [output_channel, :, 0, 0] to get all 16 input weights
                    ch_weights = param[out_idx, :, 0, 0] 
                    
                    print(f"  Channel: {ch_name}")
                    for var_idx in range(len(variables)):
                        val = ch_weights[var_idx].item()
                        print(f"    {variables[var_idx]}: {val:.6f}")
                        
    
            
            
    
    

In [9]:
print_adapter_weights('exp\exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED.pth', model = model)

<>:1: SyntaxWarning: invalid escape sequence '\e'
<>:1: SyntaxWarning: invalid escape sequence '\e'
C:\Users\perrydebussy\AppData\Local\Temp\ipykernel_28392\96752207.py:1: SyntaxWarning: invalid escape sequence '\e'
  print_adapter_weights('exp\exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED.pth', model = model)



input_adapt.0.weight: torch.Size([3, 16, 1, 1])
  Channel: R
    TMQ: 1.000000
    U850: 0.018355
    V850: -0.047000
    UBOT: 0.037738
    VBOT: -0.031979
    QREFHT: -0.027230
    PS: -0.062536
    PSL: 0.022532
    T200: -0.096299
    T500: -0.008354
    PRECT: 0.085012
    TS: -0.050598
    TREFHT: -0.089597
    Z1000: -0.014584
    Z200: 0.039663
    ZBOT: -0.111438
  Channel: G
    TMQ: 0.005158
    U850: 1.000000
    V850: 0.012362
    UBOT: 0.073369
    VBOT: 0.058110
    QREFHT: -0.009025
    PS: -0.056388
    PSL: -0.043497
    T200: -0.063970
    T500: 0.045276
    PRECT: 0.075979
    TS: -0.027323
    TREFHT: -0.001190
    Z1000: -0.007488
    Z200: -0.039979
    ZBOT: -0.064472
  Channel: B
    TMQ: 0.016963
    U850: 0.003925
    V850: 1.000000
    UBOT: 0.030162
    VBOT: -0.007588
    QREFHT: 0.020093
    PS: -0.057499
    PSL: -0.006312
    T200: 0.132974
    T500: -0.024698
    PRECT: 0.049096
    TS: -0.030419
    TREFHT: -0.112972
    Z1000: 0.082451
    Z200: 0.1

ClimateSAM(
  (ori_sam): Sam(
    (image_encoder): ImageEncoderViT(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (blocks): ModuleList(
        (0-11): 12 x Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (lin1): Linear(in_features=768, out_features=3072, bias=True)
            (lin2): Linear(in_features=3072, out_features=768, bias=True)
            (act): GELU(approximate='none')
          )
        )
      )
      (neck): Sequential(
        (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): LayerNorm2d()
        (2): Conv2d(256, 256, kernel_size=(3, 3), str

In [23]:
min(1, 8)

1

In [41]:
from model.lora_sam_dual import LoRAClimateSAMVanilla
dual_lora_model = LoRAClimateSAMVanilla(model_type='vit_b', r=64, lora_layers=None, input_weights=None, use_prompt_generator=False, mlp_ratio=1.0, freeze_base=True)
model = model.to(device=device)

print_adapter_weights('exp\exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED_NOSMOOTH.pth', model = dual_lora_model)

<>:5: SyntaxWarning: invalid escape sequence '\e'
<>:5: SyntaxWarning: invalid escape sequence '\e'
C:\Users\perrydebussy\AppData\Local\Temp\ipykernel_25956\3503349029.py:5: SyntaxWarning: invalid escape sequence '\e'
  print_adapter_weights('exp\exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED_NOSMOOTH.pth', model = dual_lora_model)


✓ Model initialized with type: vit_b
Model parameters: 106,570,795
✗ Error loading image encoder: Error(s) in loading state_dict for ImageEncoderViT:
	Missing key(s) in state_dict: "pos_embed", "patch_embed.proj.weight", "patch_embed.proj.bias", "blocks.0.norm1.weight", "blocks.0.norm1.bias", "blocks.0.attn.rel_pos_h", "blocks.0.attn.rel_pos_w", "blocks.0.attn.qkv.qkv.weight", "blocks.0.attn.qkv.qkv.bias", "blocks.0.attn.qkv.a_q_tc.weight", "blocks.0.attn.qkv.b_q_tc.weight", "blocks.0.attn.qkv.a_v_tc.weight", "blocks.0.attn.qkv.b_v_tc.weight", "blocks.0.attn.qkv.a_q_ar.weight", "blocks.0.attn.qkv.b_q_ar.weight", "blocks.0.attn.qkv.a_v_ar.weight", "blocks.0.attn.qkv.b_v_ar.weight", "blocks.0.attn.proj.weight", "blocks.0.attn.proj.bias", "blocks.0.norm2.weight", "blocks.0.norm2.bias", "blocks.0.mlp.lin1.weight", "blocks.0.mlp.lin1.bias", "blocks.0.mlp.lin2.weight", "blocks.0.mlp.lin2.bias", "blocks.1.norm1.weight", "blocks.1.norm1.bias", "blocks.1.attn.rel_pos_h", "blocks.1.attn.rel_pos_

In [11]:
import torch 
import torch.nn as nn
model = ClimateSAM(
    model_type='vit_b',
    mlp_ratio=1,
    # enable_wandb_logging=False
).to(device=device)
variables = ['TMQ', 'U850', 'V850', 'UBOT', 'VBOT', 'QREFHT', 'PS', 'PSL', 
                    'T200', 'T500', 'PRECT', 'TS', 'TREFHT', 'Z1000', 'Z200', 'ZBOT']
channels = ['R', 'G', 'B']

first_conv = model.input_adapter.input_adapt[0]  # Assuming the first layer is a Conv2d
# with torch.no_grad():
#         # 1. Start with a clean slate (no noise)
#         nn.init.constant_(first_conv.weight, 0.0)
#         nn.init.constant_(first_conv.bias, 0.0)
        
#         # 2. Explicitly map using your ClimateDataset indices
#         # Red Channel <--- TMQ (Index 0)
#         first_conv.weight[0, 0, 0, 0] = 1.0
        
#         # Green Channel <--- U850 (Index 1)
#         first_conv.weight[1, 1, 0, 0] = 1.0
        
#         # Blue Channel <--- V850 (Index 2)
#         first_conv.weight[2, 2, 0, 0] = 1.0

for name, param in model.input_adapter.named_parameters():
            if 'weight' in name:
                print(f"\n{name}: {param.shape}") # Should be [3, 16, 1, 1]
                
                for out_idx in range(3):
                    ch_name = channels[out_idx]
                    # Access [output_channel, :, 0, 0] to get all 16 input weights
                    ch_weights = param[out_idx, :, 0, 0] 
                    
                    print(f"  Channel: {ch_name}")
                    for var_idx in range(len(variables)):
                        val = ch_weights[var_idx].item()
                        print(f"    {variables[var_idx]}: {val:.6f}")
                


input_adapt.0.weight: torch.Size([3, 16, 1, 1])
  Channel: R
    TMQ: 1.000000
    U850: 0.047023
    V850: -0.023604
    UBOT: -0.074131
    VBOT: 0.052353
    QREFHT: -0.054535
    PS: 0.001207
    PSL: -0.075568
    T200: 0.054671
    T500: -0.136560
    PRECT: -0.056245
    TS: -0.007935
    TREFHT: 0.003365
    Z1000: 0.027450
    Z200: -0.015899
    ZBOT: 0.030694
  Channel: G
    TMQ: 0.058947
    U850: 1.000000
    V850: 0.076332
    UBOT: -0.027596
    VBOT: -0.008217
    QREFHT: -0.011467
    PS: 0.028612
    PSL: -0.051237
    T200: 0.057493
    T500: -0.005614
    PRECT: -0.084256
    TS: -0.023605
    TREFHT: -0.086352
    Z1000: -0.055334
    Z200: 0.021567
    ZBOT: -0.107647
  Channel: B
    TMQ: 0.011186
    U850: -0.061825
    V850: 1.000000
    UBOT: 0.016925
    VBOT: -0.040130
    QREFHT: 0.003404
    PS: 0.110504
    PSL: 0.111067
    T200: -0.081504
    T500: -0.042239
    PRECT: 0.058886
    TS: -0.004463
    TREFHT: -0.052368
    Z1000: -0.004170
    Z200: -0.

In [7]:
print_adapter_weights('exp\exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED_NOSMOOTH.pth', model = model)


input_adapt.0.weight: torch.Size([3, 16, 1, 1])
  Channel: R
    TMQ: 1.000000
    U850: 0.019864
    V850: -0.082536
    UBOT: -0.029831
    VBOT: 0.004291
    QREFHT: 0.075659
    PS: 0.066121
    PSL: -0.001752
    T200: 0.057138
    T500: -0.030049
    PRECT: -0.008565
    TS: 0.038690
    TREFHT: -0.026155
    Z1000: 0.063450
    Z200: 0.031853
    ZBOT: -0.020164
  Channel: G
    TMQ: -0.001064
    U850: 1.000000
    V850: -0.144148
    UBOT: -0.031551
    VBOT: 0.095217
    QREFHT: -0.060123
    PS: -0.007835
    PSL: -0.035083
    T200: -0.042224
    T500: 0.107040
    PRECT: 0.064899
    TS: 0.081742
    TREFHT: -0.016944
    Z1000: 0.049227
    Z200: -0.013725
    ZBOT: -0.028019
  Channel: B
    TMQ: -0.034244
    U850: 0.061199
    V850: 1.000000
    UBOT: 0.000349
    VBOT: 0.038466
    QREFHT: -0.018285
    PS: 0.088249
    PSL: -0.036078
    T200: 0.050346
    T500: -0.029636
    PRECT: -0.016313
    TS: 0.010947
    TREFHT: -0.045419
    Z1000: 0.041700
    Z200: 0.020

<>:1: SyntaxWarning: invalid escape sequence '\e'
<>:1: SyntaxWarning: invalid escape sequence '\e'
C:\Users\perrydebussy\AppData\Local\Temp\ipykernel_28392\4032618302.py:1: SyntaxWarning: invalid escape sequence '\e'
  print_adapter_weights('exp\exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED_NOSMOOTH.pth', model = model)
C:\Users\perrydebussy\AppData\Local\Temp\ipykernel_28392\4032618302.py:1: SyntaxWarning: invalid escape sequence '\e'
  print_adapter_weights('exp\exp_infused_token_vit_b_1.0_infused_token_vit_b_mlp1_CORRECTED_NOSMOOTH.pth', model = model)


NameError: name 'phase_1_checkpoint' is not defined